<a href="https://colab.research.google.com/github/e3la/i2dc/blob/main/i2dc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Welcome to i2dc: An Instagram to Digital Commons Tool!</h1>
<h2>This code is the result of vibe coding and was built mostly using Gemini.</h2>

This Colab notebook transforms an Instagram ZIP archive into structured ZIP packages, crafted for easier uploading to an institutional repository on the Digital Commons platform.

You will be guided through a simple 3-step process. Just press the ▶️ (play) button on the left of each cell, starting with Step 1. If you want to see what the code is doing you can open it below, but know, the person who vibe coded this didn't write it or read it, that was gemini. I just run it and see what happens.

---

In [ ]:
#@title <h1> **Step 1: Upload and Prepare Your Package**

# @markdown Run the cell below. It will ask you to provide your Instagram archive, either by uploading it directly or by connecting to Google Drive.

# @markdown The script will then:
# @markdown 1. Get your Instagram `.zip` file.
# @markdown 2. Extract its contents in the background.
# @markdown 3. Automatically find your Instagram username to use in the export.

from google.colab import files, drive
import os
import shutil
import zipfile
import json
import re

# This variable will be used by the next cells
zip_filepath = None

def setup_zip_file_interactive():
    """
    Interactively asks the user how they want to provide their Instagram archive ZIP file.
    """
    while True:
        print("\n--- 📂 How would you like to provide the Instagram ZIP file? ---")
        method = input(
            "1. Upload from my computer\n"
            "2. Use Google Drive\n"
            "Enter choice (1 or 2): "
        ).strip()

        if method == '1':
            print("\nPlease click 'Choose Files' and select your Instagram ZIP archive.")
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
                if filename.lower().endswith('.zip'):
                    print(f"✔️ Successfully uploaded: {filename}")
                    return os.path.join('/content', filename)
                else:
                    print(f"❌ ERROR: The uploaded file '{filename}' is not a ZIP file. Please try again.")
            else:
                print("❌ No file was uploaded. Please try again.")

        elif method == '2':
            print("\nSelected: Use Google Drive.")
            print("Connecting to your Google Drive...")
            drive.mount('/content/drive', force_remount=True)
            gdrive_path = "/content/drive/MyDrive/i2dc/"
            print(f"Searching for .zip files in your Google Drive at: '{gdrive_path}'")

            if not os.path.isdir(gdrive_path):
                print(f"❌ ERROR: The folder '{gdrive_path}' was not found.")
                print("Please create a folder named 'i2dc' in your 'My Drive' and place your ZIP file inside it, then run this cell again.")
                return None

            zip_files_found = [os.path.join(gdrive_path, f) for f in os.listdir(gdrive_path) if f.lower().endswith('.zip')]

            if not zip_files_found:
                print(f"❌ No .zip files found in '{gdrive_path}'. Please add your file and try again.")
                return None
            elif len(zip_files_found) == 1:
                chosen_filepath = zip_files_found[0]
                print(f"✔️ Found one ZIP file: '{os.path.basename(chosen_filepath)}'")
                return chosen_filepath
            else:
                print(f"\nMultiple .zip files found. Please choose one:")
                while True:
                    for i, filepath in enumerate(zip_files_found):
                        print(f"  {i+1}. {os.path.basename(filepath)}")
                    try:
                        choice_str = input(f"Enter the number of the file you want to use (1-{len(zip_files_found)}): ")
                        choice_int = int(choice_str)
                        if 1 <= choice_int <= len(zip_files_found):
                            chosen_filepath = zip_files_found[choice_int - 1]
                            print(f"✔️ You selected: '{os.path.basename(chosen_filepath)}'")
                            return chosen_filepath
                        else:
                            print(f"❌ Invalid number. Please enter a number between 1 and {len(zip_files_found)}.")
                    except ValueError:
                        print("❌ Invalid input. Please enter a number.")
        else:
            print("❌ Invalid choice. Please enter 1 or 2.")

def extract_instagram_handle(personal_info_path):
    try:
        with open(personal_info_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        username_value = None
        if isinstance(data, dict) and "profile_user" in data and isinstance(data["profile_user"], list) and len(data["profile_user"]) > 0:
            profile_info = data["profile_user"][0]
            if isinstance(profile_info, dict) and "string_map_data" in profile_info and isinstance(profile_info["string_map_data"], dict):
                string_data = profile_info["string_map_data"]
                if "Username" in string_data and isinstance(string_data["Username"], dict) and "value" in string_data["Username"]:
                    username_value = string_data["Username"]["value"]
        if username_value:
            return username_value
        else:
            print("  -> Username key not found in the expected structure within the file.")
            return "unknown_user"
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"  -> Could not read or parse the JSON file. Reason: {e}")
        return "unknown_user"

BASE_DIR = "/content"
EXTRACTED_DATA_DIR = os.path.join(BASE_DIR, "extracted_data")
instagram_handle = "unknown_user"

zip_filepath = setup_zip_file_interactive()

if zip_filepath and os.path.exists(zip_filepath):
    print(f"\n✅ File ready at: {zip_filepath}")
    print("\n⚙️ Processing your file...")
    print(f"--- 📂 Extracting {os.path.basename(zip_filepath)} ---")
    if os.path.exists(EXTRACTED_DATA_DIR):
        shutil.rmtree(EXTRACTED_DATA_DIR)
    os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(EXTRACTED_DATA_DIR)
        print("✔️ Successfully extracted archive.")
    except Exception as e:
        print(f"❌ ERROR during extraction: {e}")

    # Find the correct path to the JSON file by checking common locations
    path_to_check = None
    path1 = os.path.join(EXTRACTED_DATA_DIR, 'your_instagram_activity', 'account_information', 'personal_information.json')
    path2 = os.path.join(EXTRACTED_DATA_DIR, 'personal_information', 'personal_information.json')
    # Add a third, specific path as a fallback, based on observed archive structures.
    path3 = os.path.join(EXTRACTED_DATA_DIR, 'personal_information', 'personal_information', 'personal_information.json')

    if os.path.exists(path1):
        path_to_check = path1
    elif os.path.exists(path2):
        path_to_check = path2
    elif os.path.exists(path3):
        path_to_check = path3

    if path_to_check:
        print(f"✔️ Found metadata file at: {path_to_check}")
        instagram_handle = extract_instagram_handle(path_to_check)
    else:
        print("❌ Could not find 'personal_information.json' in any standard location within the ZIP.")

    if instagram_handle != "unknown_user":
        print(f"\n✅ Success! Username Found: @{instagram_handle}")
        print("\n➡️ You can now proceed to Step 2.")
    else:
        print("\n⚠️ Could not automatically find your username. A placeholder will be used.")
        print("\n➡️ You can now proceed to Step 2.")
else:
    print("\n❌ Process stopped. A valid ZIP file was not provided.")
    print("Please run this cell again to provide a file.")


--- 📂 How would you like to provide the Instagram ZIP file? ---
1. Upload from my computer
2. Use Google Drive
Enter choice (1 or 2): 2

Selected: Use Google Drive.
Connecting to your Google Drive...
Mounted at /content/drive
Searching for .zip files in your Google Drive at: '/content/drive/MyDrive/i2dc/'

Multiple .zip files found. Please choose one:
  1. instagram-umsllibraries-2025-03-07-Ardjbhx1.zip
  2. reels_2025-07-04_READY_FOR_REVIEW.zip
Enter the number of the file you want to use (1-2): 1
✔️ You selected: 'instagram-umsllibraries-2025-03-07-Ardjbhx1.zip'

✅ File ready at: /content/drive/MyDrive/i2dc/instagram-umsllibraries-2025-03-07-Ardjbhx1.zip

⚙️ Processing your file...
--- 📂 Extracting instagram-umsllibraries-2025-03-07-Ardjbhx1.zip ---
✔️ Successfully extracted archive.
✔️ Found metadata file at: /content/extracted_data/personal_information/personal_information/personal_information.json

✅ Success! Username Found: @umsllibraries

➡️ You can now proceed to Step 2.


### **Step 2: Configure Your Export Settings**

Use the form in the cell below to customize the output of your export. Your username (found in Step 1) is used to show you examples.

This form lets you control:
- **Content to Process:** Choose whether to include Posts, Reels, and/or Stories.
- **Title Formatting:** Select a pre-defined title structure for your items.
- **Metadata Columns:** Customize the columns in the final Excel file.
- **Abstract Formatting:** Add a preamble and format the post text with HTML.

**Action:** Adjust the settings below, then run the cell to save your configuration.

In [ ]:
#@title Configure Your Export Settings
import os
import json

def _find_instagram_handle_for_config():
    """A helper to find the username if it wasn't found in Step 1."""
    def extract_handle(path):
        try:
            with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
            profile_list = data.get('profile_user', [])
            if profile_list:
                string_map = profile_list[0].get('string_map_data', {})
                return string_map.get('Username', {}).get('value', 'unknown_user')
            return 'unknown_user'
        except (FileNotFoundError, json.JSONDecodeError, IndexError):
            return 'unknown_user'

    print("ℹ️ Checking for Instagram username...")
    EXTRACTED_DATA_DIR = "/content/extracted_data"
    if not os.path.isdir(EXTRACTED_DATA_DIR):
        print("\n❌ CRITICAL: Instagram data has not been extracted yet.")
        print("Please run Step 1 to upload and extract your archive before running this step.")
        # This will stop the cell execution with a clear error
        raise Exception("Prerequisite Step 1 not completed.")

    path1 = os.path.join(EXTRACTED_DATA_DIR, 'your_instagram_activity', 'account_information', 'personal_information.json')
    path2 = os.path.join(EXTRACTED_DATA_DIR, 'personal_information', 'personal_information.json')
    # Add a third, specific path as a fallback, based on observed archive structures.
    path3 = os.path.join(EXTRACTED_DATA_DIR, 'personal_information', 'personal_information', 'personal_information.json')

    path_to_check = None
    if os.path.exists(path1): path_to_check = path1
    elif os.path.exists(path2): path_to_check = path2
    elif os.path.exists(path3): path_to_check = path3


    if path_to_check:
        handle = extract_handle(path_to_check)
        if handle != 'unknown_user':
            print(f"✔️ Success! Found username: @{handle}")
            return handle

    print("⚠️ Could not find username automatically. Using a placeholder.")
    return 'unknown_user'

# --- Main logic for this cell ---
# Check if the handle is already known, if not, try to find it.
if 'instagram_handle' not in globals() or instagram_handle == 'unknown_user':
    instagram_handle = _find_instagram_handle_for_config()

# Use the found handle for examples, or a placeholder if it's still unknown.
ig_handle_for_examples = instagram_handle if instagram_handle != 'unknown_user' else "your_username"

print("\n--- Title Format Examples ---")
print(f"1. Default: Instagram Post by {ig_handle_for_examples} on 2024-08-26")
print(f"2. Simple:  {ig_handle_for_examples} | Post | 2024-08-26")
print(f"3. Alt:     Post by {ig_handle_for_examples} (2024-08-26)")
print("-" * 50)

#@markdown ### **1. Content to Process**
#@markdown Select which types of Instagram content you want to include in the export.
process_posts = True #@param {type:"boolean"}
process_reels = False #@param {type:"boolean"}
process_stories = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### **2. Title Formatting**
#@markdown Choose a title format from the examples printed above.
title_format_choice = "User | Type | Date" #@param ["Default (Type by User on Date)", "User | Type | Date", "Type by User (Date)", "Date - User - Type", "Custom Format..."]
#@markdown If you chose "Custom Format...", define it below using `{username}`, `{doc_type}`, `{doc_type_short}`, and `{date}`.
custom_title_template = "" #@param {type:"string"}

#@markdown ---
#@markdown ### **3. Metadata Columns**
#@markdown Enter the metadata columns for the final Excel file, separated by commas.
#@markdown If you want them, add a comma and any of these: `like_count`, `comments_disabled`, `latitude`, `longitude`, `original_filename`, `source_file_path`
metadata_columns_str = "title, publication_date, abstract, keywords, document_type, fulltext_url, additional_files, instagram_username" #@param {type:"string"}

#@markdown ---
#@markdown ### **4. Abstract Formatting**
#@markdown Choose whether to add a `<b>Posted Text: </b>` preamble to the abstract field. This option also converts the abstract (your original caption) into HTML, properly formatting newlines and emojis.
add_abstract_preamble = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### **5. Output & Batching**
#@markdown Choose where to save the final ZIP packages and how many items to include in each.
save_to_gdrive = True #@param {type:"boolean"}
#@markdown Set the maximum number of primary media items (Posts, Reels, or Stories) per ZIP file. SRT files do not count towards this limit.
batch_size = 20 #@param {type:"number"}
#@markdown I recommend 20 for posts if you're wanting to generate AI alt text for them with https://aistudio.google.com/app/prompts?state=%7B%22ids%22:%5B%221zgUh4dqoGWewtqPE40gBb96Vt2TMujgA%22%5D,%22action%22:%22open%22,%22userId%22:%22106905111806513074045%22,%22resourceKeys%22:%7B%7D%7D&usp=sharin
" target="_blank>"https://aistudio.google.com/app/prompts?state=%7B%22ids%22:%5B%221zgUh4dqoGWewtqPE40gBb96Vt2TMujgA%22%5D,%22action%22:%22open%22,%22userId%22:%22106905111806513074045%22,%22resourceKeys%22:%7B%7D%7D&usp=sharing"

#@markdown ---
#@markdown ### **6. Filename Customization**
#@markdown Add a custom tag to the end of each ZIP filename (e.g., `_myproject`). The format will be `[type]_[date]_[custom_text].zip`.
custom_filename_append = "_READY_FOR_REVIEW" #@param {type:"string"}


print("\n✅ Configuration loaded.")
print("\n➡️ All settings are ready. You can now run the final step below! ")


--- Title Format Examples ---
1. Default: Instagram Post by umsllibraries on 2024-08-26
2. Simple:  umsllibraries | Post | 2024-08-26
3. Alt:     Post by umsllibraries (2024-08-26)
--------------------------------------------------

✅ Configuration loaded.

➡️ All settings are ready. You can now run the final step below! 


In [ ]:
# @title <h1>Step 3: Process Data and Download Packages</h1>
# @markdown This is the final step. Running this cell will process all your data according to the settings from Step 2.

# @markdown This cell will:
# @markdown - Install necessary Python libraries for processing (`ftfy`, `emoji`).
# @markdown - Clean up text and emoji encoding in your Instagram data.
# @markdown - Create separate, structured ZIP packages with custom names, potentially split into batches.
# @markdown - Save ZIPs to the main `i2dc` Google Drive folder or start a browser download.

# @markdown **Action:** Press the ▶️ play button. This may take several minutes depending on the size of your archive.

# --- Initial Setup ---
!pip install ftfy emoji pandas openpyxl -q
print("✔️ Helper libraries are installed and ready.")

# --- Core Library Imports ---
import os, shutil, zipfile, json, re
from datetime import datetime
import pandas as pd
from google.colab import files
import emoji
import math

try:
    import ftfy
    FTFY_AVAILABLE = True
except ImportError:
    FTFY_AVAILABLE = False

# --- Global Constants ---
BASE_DIR = "/content"
EXTRACTED_DATA_DIR = os.path.join(BASE_DIR, "extracted_data")
# Local output folder for browser downloads
LOCAL_OUTPUT_DIR = os.path.join(BASE_DIR, "batchup")
# Google Drive main output folder
GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/i2dc/"
INSTAGRAM_ACTIVITY_FOLDER_NAME = "your_instagram_activity"

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def extract_hashtags(text):
    if not isinstance(text, str): return ""
    hashtags = re.findall(r"#(\w+)", text)
    return ", ".join(hashtags)

def format_caption_for_html(text):
    if not isinstance(text, str): return ""
    def replace_shortcode(match):
        shortcode = match.group(0)
        original_emoji = emoji.emojize(shortcode, language='alias')
        if original_emoji == shortcode: return shortcode
        aria_label = match.group(1).replace('_', ' ')
        return f'<span role="img" aria-label="{aria_label}">{original_emoji}</span>'
    demojized_text = emoji.demojize(text, language='alias')
    html_text = re.sub(r':([a-zA-Z0-9_+-]+(?:_skin_tone)?:)', replace_shortcode, demojized_text)
    html_text = html_text.replace('\n', '<br>\n')
    return html_text

def fix_json_encoding(media_json_dir):
    if not FTFY_AVAILABLE: return
    print("\n--- 🔎 Scanning and fixing text encoding in JSON files ---")
    if not os.path.isdir(media_json_dir):
        print(f"❌ ERROR: Media JSON directory not found at: {media_json_dir}")
        return
    total_files_changed, total_fields_fixed = 0, 0
    def _fix_text_in_obj(obj, key):
        nonlocal total_fields_fixed
        original_text = obj.get(key)
        if isinstance(original_text, str) and original_text:
            fixed_text = ftfy.fix_text(original_text)
            if fixed_text != original_text: obj[key] = fixed_text; total_fields_fixed += 1; return True
        return False
    for filename in os.listdir(media_json_dir):
        if not filename.lower().endswith('.json'): continue
        json_path = os.path.join(media_json_dir, filename)
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            file_was_changed = False
            data_list = []
            if 'posts_1.json' in filename: data_list = data if isinstance(data, list) else []
            elif 'reels.json' in filename: data_list = data.get('ig_reels_media', [])
            elif 'stories.json' in filename: data_list = data.get('ig_stories', [])
            for item in data_list:
                if 'posts' in filename: media_items = item.get('media', [])
                else: media_items = item.get('media', [item])
                if isinstance(item, dict) and _fix_text_in_obj(item, 'title'): file_was_changed = True
                for media_item in media_items:
                    if isinstance(media_item, dict) and _fix_text_in_obj(media_item, 'title'): file_was_changed = True
            if file_was_changed:
                total_files_changed += 1
                with open(json_path, 'w', encoding='utf-8') as f: json.dump(data, f, ensure_ascii=False, indent=2)
        except Exception:
            pass # Ignore errors on files that don't match structure
    print(f"✔️ Text fixing complete. Total fields fixed: {total_fields_fixed} in {total_files_changed} files.")

# ==============================================================================
# CORE PROCESSING FUNCTION
# ==============================================================================
def process_media_type(media_type, json_filename, username, selected_columns, title_format_choice_tuple, add_abstract_preamble_flag, save_to_gdrive_flag, batch_size_limit, creation_date_str, filename_append_str):
    print(f"\n{'='*20} PROCESSING: {media_type.upper()} {'='*20}")

    batch_size_limit = max(1, int(batch_size_limit))
    staging_dir = os.path.join(EXTRACTED_DATA_DIR, f'{media_type}_export_staging')
    if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
    os.makedirs(staging_dir, exist_ok=True)

    json_path = os.path.join(EXTRACTED_DATA_DIR, INSTAGRAM_ACTIVITY_FOLDER_NAME, 'media', json_filename)
    try:
        with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"ℹ️ No data file found for '{media_type}' ({json_filename}). Skipping.")
        return

    items_to_process = []
    if media_type == 'posts': items_to_process = data
    elif media_type == 'reels': items_to_process = data.get('ig_reels_media', [])
    elif media_type == 'stories': items_to_process = data.get('ig_stories', [])
    if not items_to_process:
        print(f"ℹ️ No items found for '{media_type}'. Nothing to process.")
        return

    items_to_process.sort(key=lambda x: x.get('creation_timestamp', float('inf')))
    print(f"✔️ Found {len(items_to_process)} total {media_type} items to analyze.")
    excel_data_rows, skipped_counts = [], {'no_uri': 0, 'file_missing': 0, 'copy_error': 0}

    for item_index, item in enumerate(items_to_process):
        media_list = item.get('media', [item])
        post_level_timestamp = item.get('creation_timestamp', media_list[0].get('creation_timestamp'))
        if not post_level_timestamp: continue
        post_level_caption = item.get('title', media_list[0].get('title', ''))
        for media_index, media_item in enumerate(media_list):
            if not isinstance(media_item, dict): continue
            original_uri = media_item.get('uri')
            if not original_uri or original_uri.startswith(('http://', 'https://')): skipped_counts['no_uri'] += 1; continue
            source_media_path = os.path.join(EXTRACTED_DATA_DIR, original_uri)
            if not os.path.exists(source_media_path): skipped_counts['file_missing'] += 1; continue
            date_obj = datetime.fromtimestamp(post_level_timestamp)
            original_filename_base = os.path.basename(original_uri)
            new_media_filename = f"instagram_{username}_{media_type}_{date_obj.strftime('%Y-%m-%d')}_{item_index+1}_{media_index+1}{os.path.splitext(original_uri)[-1]}"
            dest_media_path = os.path.join(staging_dir, new_media_filename)
            try: shutil.copy2(source_media_path, dest_media_path)
            except Exception: skipped_counts['copy_error'] += 1; continue
            additional_files_value = ''
            if media_type == 'reels':
                subtitles_uri = media_item.get('media_metadata', {}).get('video_metadata', {}).get('subtitles', {}).get('uri')
                if subtitles_uri and os.path.exists(os.path.join(EXTRACTED_DATA_DIR, subtitles_uri)):
                    source_srt_path = os.path.join(EXTRACTED_DATA_DIR, subtitles_uri)
                    new_srt_filename = os.path.splitext(new_media_filename)[0] + '.srt'
                    dest_srt_path = os.path.join(staging_dir, new_srt_filename)
                    try: shutil.copy2(source_srt_path, dest_srt_path); additional_files_value = new_srt_filename
                    except Exception as e: print(f"  ⚠️ Warning: Could not copy subtitle file {os.path.basename(source_srt_path)}. Error: {e}")
            formatted_caption = format_caption_for_html(post_level_caption)
            final_abstract = f"<b>Posted Text: </b><br>{formatted_caption}" if add_abstract_preamble_flag else formatted_caption
            row_data = {'title': "Placeholder Title", 'publication_date': date_obj.date(), 'abstract': final_abstract, 'keywords': extract_hashtags(post_level_caption),'document_type': f"Instagram {media_type.capitalize()}", 'fulltext_url': new_media_filename, 'instagram_username': username, 'creation_timestamp': post_level_timestamp, 'like_count': media_item.get('like_count', ''), 'comments_disabled': item.get('comments_disabled', False), 'latitude': media_item.get('location', {}).get('lat', ''), 'longitude': media_item.get('location', {}).get('lng', ''), 'original_filename': original_filename_base, 'source_file_path': original_uri, 'additional_files': additional_files_value}
            excel_data_rows.append(row_data)

    if not excel_data_rows: print(f"ℹ️ No processable media items found for {media_type}."); return
    df = pd.DataFrame(excel_data_rows)
    choice, custom_template = title_format_choice_tuple
    template_map = {'Default (Type by User on Date)': "{doc_type} by {username} on {date}", 'User | Type | Date': "{username} | {doc_type_short} | {date}", 'Type by User (Date)': "{doc_type_short} by {username} ({date})", 'Date - User - Type': "{date} - {username} - {doc_type_short}", 'Custom Format...': custom_template or "{doc_type} by {username} on {date}"}
    template_str = template_map.get(choice)
    df['doc_type_short'] = df['document_type'].str.replace("Instagram ", "")
    df['date_str'] = df['publication_date'].apply(lambda d: d.strftime('%Y-%m-%d'))
    df['title'] = df.apply(lambda row: template_str.format(username=row['instagram_username'], doc_type=row['document_type'], doc_type_short=row['doc_type_short'], date=row['date_str']), axis=1)
    df['_date_rank'] = df.groupby('date_str').cumcount() + 1
    date_counts = df['date_str'].value_counts().to_dict()
    df['title'] = df.apply(lambda row: f"{row['title']} - {row['_date_rank']} of {date_counts[row['date_str']]}" if date_counts[row['date_str']] > 1 else row['title'], axis=1)
    df.drop(columns=['doc_type_short', 'date_str', '_date_rank', 'creation_timestamp'], inplace=True, errors='ignore')
    final_df_full = pd.DataFrame(df, columns=[col for col in selected_columns if col in df.columns])

    num_batches = math.ceil(len(final_df_full) / batch_size_limit)
    output_base_dir = GDRIVE_OUTPUT_DIR if save_to_gdrive_flag else LOCAL_OUTPUT_DIR
    os.makedirs(output_base_dir, exist_ok=True)

    for i in range(num_batches):
        batch_num = i + 1
        print(f"\n--- Creating Batch {batch_num} of {num_batches} for {media_type.capitalize()} ---")
        start_index = i * batch_size_limit
        end_index = start_index + batch_size_limit
        batch_df = final_df_full.iloc[start_index:end_index]

        # --- New Filename Generation Logic ---
        filename_parts = [media_type, creation_date_str]
        if filename_append_str:
             # a little cleanup to remove leading/trailing underscores if user adds them
            filename_parts.append(filename_append_str.strip('_'))

        base_filename = "_".join(filename_parts)
        excel_filename = f"{base_filename}_metadata.xlsx"
        zip_filename = f"{base_filename}.zip"
        if num_batches > 1:
            zip_filename = f"{base_filename}_batch_{batch_num}.zip"
            excel_filename = f"{base_filename}_batch_{batch_num}_metadata.xlsx"
        # --- End New Filename Logic ---

        excel_path = os.path.join(staging_dir, excel_filename)
        batch_df.to_excel(excel_path, index=False, engine='openpyxl')
        print(f"✔️ Metadata for {len(batch_df)} items written to {excel_filename}")
        final_zip_path = os.path.join(output_base_dir, zip_filename)
        with zipfile.ZipFile(final_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(excel_path, arcname=excel_filename)
            for row in batch_df.itertuples():
                media_file_path = os.path.join(staging_dir, row.fulltext_url)
                if os.path.exists(media_file_path): zipf.write(media_file_path, arcname=row.fulltext_url)
                if hasattr(row, 'additional_files') and row.additional_files:
                    srt_file_path = os.path.join(staging_dir, row.additional_files)
                    if os.path.exists(srt_file_path): zipf.write(srt_file_path, arcname=row.additional_files)
        if save_to_gdrive_flag:
            print(f"✔️ Final package '{zip_filename}' saved to your Google Drive at: {output_base_dir}")
        else:
            print(f"✔️ Final package '{zip_filename}' is ready. Triggering download...")
            files.download(final_zip_path)

# ==============================================================================
# MAIN EXECUTION SCRIPT
# ==============================================================================
def main():
    print("=" * 60)
    print("🚀 Starting the Instagram to Digital Commons Exporter! 🚀")
    print("=" * 60)
    if 'zip_filepath' not in globals() or not zip_filepath: print("\n❌ ABORTED: Please complete Step 1 first."); return
    if 'instagram_handle' not in globals(): print("\n❌ ABORTED: Username not found. Please complete Step 1."); return
    selected_cols = [col.strip() for col in metadata_columns_str.split(',') if col.strip()]
    title_choice_tuple = (title_format_choice, custom_title_template)
    creation_date_str = datetime.now().strftime('%Y-%m-%d')

    output_dir = GDRIVE_OUTPUT_DIR if save_to_gdrive else LOCAL_OUTPUT_DIR
    if os.path.exists(output_dir):
        # Safer cleanup: Only remove zips that look like they were generated by this script.
        print(f"🧹 Cleaning up previous output files from '{output_dir}'...")
        for f in os.listdir(output_dir):
            if (f.startswith(('posts_', 'reels_', 'stories_'))) and f.endswith('.zip'):
                try:
                    os.remove(os.path.join(output_dir, f))
                except Exception as e:
                    print(f"  ⚠️ Could not remove old file: {f}. Reason: {e}")
    os.makedirs(output_dir, exist_ok=True)

    print("--- Settings Confirmed ---")
    print(f"Username: @{instagram_handle}")
    print(f"Processing Posts: {process_posts}, Reels: {process_reels}, Stories: {process_stories}")
    print(f"Save to Google Drive: {save_to_gdrive}")
    print(f"Items per ZIP file: {batch_size}")
    print(f"Custom Filename Tag: '{custom_filename_append}'")
    print("-" * 26)

    media_json_dir = os.path.join(EXTRACTED_DATA_DIR, INSTAGRAM_ACTIVITY_FOLDER_NAME, 'media')
    fix_json_encoding(media_json_dir)

    process_args = {
        "username": instagram_handle, "selected_columns": selected_cols, "title_format_choice_tuple": title_choice_tuple,
        "add_abstract_preamble_flag": add_abstract_preamble, "save_to_gdrive_flag": save_to_gdrive,
        "batch_size_limit": batch_size, "creation_date_str": creation_date_str,
        "filename_append_str": custom_filename_append
    }
    if process_posts: process_media_type('posts', 'posts_1.json', **process_args)
    if process_reels: process_media_type('reels', 'reels.json', **process_args)
    if process_stories: process_media_type('stories', 'stories.json', **process_args)

    print("\n🎉 All processing complete! 🎉")
    if save_to_gdrive:
        print(f"Check your main 'i2dc' folder in Google Drive for the final ZIP packages.")
    else:
        print(f"Check your browser downloads for the final ZIP packages.")
        print(f"A copy is also saved in the '{os.path.basename(output_dir)}' folder in the Files panel on the left.")

# Run the main function
main()

✔️ Helper libraries are installed and ready.
🚀 Starting the Instagram to Digital Commons Exporter! 🚀
🧹 Cleaning up previous output files from '/content/drive/MyDrive/i2dc/'...
--- Settings Confirmed ---
Username: @umsllibraries
Processing Posts: True, Reels: False, Stories: False
Save to Google Drive: True
Items per ZIP file: 20
Custom Filename Tag: '_READY_FOR_REVIEW'
--------------------------

--- 🔎 Scanning and fixing text encoding in JSON files ---
✔️ Text fixing complete. Total fields fixed: 262 in 3 files.

==================== PROCESSING: POSTS ====================
✔️ Found 646 total posts items to analyze.

--- Creating Batch 1 of 43 for Posts ---
✔️ Metadata for 20 items written to posts_2025-07-05_READY_FOR_REVIEW_batch_1_metadata.xlsx
✔️ Final package 'posts_2025-07-05_READY_FOR_REVIEW_batch_1.zip' saved to your Google Drive at: /content/drive/MyDrive/i2dc/

--- Creating Batch 2 of 43 for Posts ---
✔️ Metadata for 20 items written to posts_2025-07-05_READY_FOR_REVIEW_batch_